# 04 - LangSmith Tracing Setup

This notebook validates LangSmith credentials, sends traced inference traffic, and builds local charts from run metadata.

It expects:
- `LANGSMITH_API_KEY` (or `LANGCHAIN_API_KEY`) in your notebook environment
- proxy endpoint reachable at `http://localhost:8000/ollama/api/chat`


In [ ]:
import os
import sys
from pathlib import Path

expected = "/usr/local/bin/python3.11"
print(f"Python executable: {sys.executable}")
print(f"Python version   : {sys.version.split()[0]}")
if Path(sys.executable).resolve().as_posix() != expected:
    print(f"WARNING: This notebook is designed for {expected}.")
    print("Switch the notebook kernel to Python 3.11 before continuing.")
else:
    print("Kernel check passed.")


In [ ]:
import importlib
import subprocess
import sys

packages = ["requests", "pandas", "matplotlib", "pyyaml", "langsmith", "tabulate"]
missing = []
for pkg in packages:
    module_name = "yaml" if pkg == "pyyaml" else pkg
    try:
        importlib.import_module(module_name)
    except Exception:
        missing.append(pkg)

if missing:
    print("Installing missing packages:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
else:
    print("All common packages are already installed.")


In [ ]:
import os
from datetime import datetime, timedelta, timezone

import matplotlib.pyplot as plt
import pandas as pd
import requests
from langsmith import Client

LANGSMITH_ENDPOINT = os.getenv("LANGSMITH_ENDPOINT", "https://api.smith.langchain.com")
LANGSMITH_PROJECT = os.getenv("LANGSMITH_PROJECT", "k3s-ollama-gemma-local")
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY") or os.getenv("LANGCHAIN_API_KEY")
PROXY_CHAT_URL = "http://localhost:8000/ollama/api/chat"
MODEL_NAME = os.getenv("OLLAMA_MODEL", "gemma3-1b-it-gguf-local")


def url_available(url: str, timeout: int = 3) -> bool:
    try:
        r = requests.get(url, timeout=timeout)
        return r.status_code < 500
    except Exception:
        return False


PROXY_AVAILABLE = url_available("http://localhost:8000/healthz")
client = None
if not LANGSMITH_API_KEY:
    print("LANGSMITH_API_KEY/LANGCHAIN_API_KEY is not set. LangSmith API cells will run in skip mode.")
else:
    try:
        client = Client(api_url=LANGSMITH_ENDPOINT, api_key=LANGSMITH_API_KEY)
    except Exception as exc:
        print(f"Unable to initialize LangSmith client: {exc}")

print("LangSmith endpoint:", LANGSMITH_ENDPOINT)
print("LangSmith project :", LANGSMITH_PROJECT)
print("PROXY_AVAILABLE   :", PROXY_AVAILABLE)
print("LANGSMITH_READY   :", client is not None)


## Verify Project Access


In [ ]:
project_names = []
if client is None:
    print("Skipping project listing: LangSmith client is not configured.")
else:
    try:
        projects = list(client.list_projects(limit=50))
        project_names = [p.name for p in projects]
        print(f"Visible projects: {len(project_names)}")
        print("Target project present:", LANGSMITH_PROJECT in project_names)
    except Exception as exc:
        print(f"Project listing failed: {exc}")

project_names[:15]


## Generate Traced Inference Samples via Proxy


In [ ]:
prompts = [
    "What does observability mean for LLM inference?",
    "Give a one paragraph summary of LangSmith runs.",
    "Why include status code in traces?",
    "How can I reduce memory in a local k3s stack?",
]

sent = []
if not PROXY_AVAILABLE:
    print("Skipping traced proxy sample generation: proxy endpoint is not reachable on localhost:8000")
else:
    for i in range(10):
        prompt = prompts[i % len(prompts)]
        body = {
            "model": MODEL_NAME,
            "stream": False,
            "messages": [{"role": "user", "content": prompt}],
        }
        try:
            r = requests.post(PROXY_CHAT_URL, json=body, timeout=180)
            sent.append({"index": i + 1, "status": r.status_code, "prompt": prompt})
        except Exception as exc:
            sent.append({"index": i + 1, "status": -1, "prompt": prompt, "error": str(exc)})

pd.DataFrame(sent)


## Pull Recent Runs From LangSmith


In [ ]:
runs_df = pd.DataFrame(columns=["run_id", "name", "run_type", "start_time", "error", "status_code", "latency_ms", "tags"])

if client is None:
    print("Skipping run fetch: LangSmith client is not configured.")
else:
    try:
        cutoff = datetime.now(timezone.utc) - timedelta(hours=2)
        runs = list(client.list_runs(project_name=LANGSMITH_PROJECT, start_time=cutoff, limit=200))
        print("Runs fetched:", len(runs))

        rows = []
        for run in runs:
            outputs = getattr(run, "outputs", {}) or {}
            rows.append(
                {
                    "run_id": str(getattr(run, "id", "")),
                    "name": getattr(run, "name", ""),
                    "run_type": getattr(run, "run_type", ""),
                    "start_time": getattr(run, "start_time", None),
                    "error": getattr(run, "error", None),
                    "status_code": outputs.get("status_code"),
                    "latency_ms": outputs.get("latency_ms"),
                    "tags": ",".join(getattr(run, "tags", []) or []),
                }
            )

        runs_df = pd.DataFrame(rows)
    except Exception as exc:
        print(f"Run fetch failed: {exc}")

runs_df.head(20)


In [ ]:
if not runs_df.empty:
    runs_df["is_error"] = runs_df["error"].notna()
    numeric_latency = pd.to_numeric(runs_df["latency_ms"], errors="coerce")
    print("Error runs:", int(runs_df["is_error"].sum()))
    print("Latency rows:", int(numeric_latency.notna().sum()))

    chart_df = runs_df.copy()
    chart_df = chart_df[pd.to_numeric(chart_df["latency_ms"], errors="coerce").notna()].copy()
    chart_df["latency_ms"] = pd.to_numeric(chart_df["latency_ms"])
    chart_df = chart_df.sort_values("start_time").tail(50)

    if not chart_df.empty:
        ax = chart_df.plot(x="start_time", y="latency_ms", figsize=(10, 4), marker="o", title="LangSmith Latency (Recent Runs)")
        ax.set_ylabel("Latency (ms)")
        plt.show()


In [ ]:
# Quick tag breakdown to validate open-webui/proxy activity
if runs_df.empty:
    print("No runs available for tag breakdown.")
else:
    tag_counts = (
        runs_df["tags"]
        .fillna("")
        .str.split(",")
        .explode()
        .str.strip()
    )
    tag_counts = tag_counts[tag_counts != ""]
    tag_counts.value_counts().head(20)


## Done
Open `smith.langchain.com`, choose project **`k3s-ollama-gemma-local`**, and compare dashboard charts with the notebook output.
